# 23. Pareamento com todos os pontos do cadastro municipal

Comparamos os 1.180.102 registros do CNEFE com os 757.113 pontos do cadastro municipal. As análises usam descrições de endereço, coordenadas, vínculos fiscais e camadas territoriais.

1. Como os registros e pontos se condensam em chaves de logradouro e número?
2. Quanto do CNEFE encontra par pela chave e a que distância?
3. Que padrões cadastrais ocorrem nas chaves exclusivas?
4. Que condições estão presentes nos pares a mais de 100 m?
5. Quanto a geometria de referência muda a distância medida?
6. O que acontece com a cauda e a cobertura quando bairro e CEP são exigidos?

As entradas são as bases processadas, o cadastro imobiliário, os lotes do CTM, os eixos de logradouro e a camada de assentamentos. As tabelas saem em `outputs/referencia_completa/tables`, e o pareamento em `cnefe_identidade_completa.parquet`.


## 1. Bases

O CNEFE tem 1.180.102 registros e o BHMap 757.113 pontos, ambos em EPSG:31983. Todos os pontos municipais são incluídos na referência.


In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt
from rapidfuzz import fuzz
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.metrics import chave_identidade, separar_logradouro_numero, conciliar_logradouros, parear_por_identidade
from src.normalize import normalize_logradouro
P = config.PROCESSED_DIR
OUT = ROOT/'outputs/referencia_completa'
T, F = OUT/'tables', OUT/'figures'
for d in (T, F): d.mkdir(parents=True, exist_ok=True)
SEED, TOL = config.SEED, 10

C = gpd.read_parquet(P/'cnefe_bh.parquet')
B = gpd.read_parquet(P/'bhmap_bh.parquet', columns=['IDEND', 'SIGLA_TIPO', 'NOME_LOGRA', 'NUMERO_IMO', 'NOME_BAIRR', 'NOME_REGIO', 'CEP', 'geometry'])
arquivo_iptu = config.RAW_DIR/'bhmap_context'/'CADASTRO_IMOBILIARIO.csv'
iptu = pd.read_csv(arquivo_iptu, usecols=['NULOTCTM', 'TIPO_LOGRADOURO', 'NOME_LOGRADOURO', 'NUMERO_IMOVEL'], dtype=str)
lotes = gpd.read_parquet(P/'lote_ctm.parquet', columns=['NULOTCTM', 'AREA_M2', 'geometry']).drop_duplicates('NULOTCTM').set_index('NULOTCTM')
vila = gpd.read_parquet(P/'vila_favela.parquet', columns=['geometry'])
assert C.crs == B.crs == lotes.crs == vila.crs
assert C.index.is_unique and B.index.is_unique and len(C) == 1180102 and len(B) == 757113
uniao_vila = vila.geometry.union_all()
em_vila = pd.Series(shapely.within(C.geometry.to_numpy(), uniao_vila), index=C.index)
em_vila_b = pd.Series(shapely.within(B.geometry.to_numpy(), uniao_vila), index=B.index)
apto = pd.to_numeric(C.COD_TIPO_ESPECI, errors='coerce').eq(103).fillna(False)
estab = pd.to_numeric(C.COD_ESPECIE, errors='coerce').isin([3, 4, 5, 6, 8]).fillna(False)
assert int(em_vila.sum()) == 128149 and not (apto & estab).any()
print(f'CNEFE: {len(C):,} registros | BHMap: {len(B):,} pontos | IPTU: {len(iptu):,} unidades | lotes: {len(lotes):,}')

CNEFE: 1,180,102 registros | BHMap: 757,113 pontos | IPTU: 896,964 unidades | lotes: 365,443


## 2. Chaves de logradouro e número

No CNEFE, logradouro e número vêm num único campo (`LOGRAD_NUM`). O corte é feito na última vírgula, e recua para a anterior quando o trecho final não começa por dígito, o que indica complemento. No BHMap, tipo, nome e número vêm em campos separados, e o tipo abreviado é unido ao nome. As duas bases passam pela mesma forma canônica, sem acento, em caixa alta e com o tipo por extenso. Do número fica a primeira sequência de dígitos, de modo que 210 e 210 A formam a mesma chave. Os nomes do CNEFE ausentes dos nomes do BHMap são conciliados por token sort ratio, com limiar 90 e margem de 2 pontos sobre o segundo candidato. Os nomes do IPTU passam pela mesma conciliação, e a abreviação BEC é expandida para BECO antes dela.

In [2]:
def chave(log, num):
    return (log.astype('string')+'|'+num.astype('string')).where(log.notna() & log.ne('') & num.notna())
def chaves_bhmap(df):
    l, n = chave_identidade(df.SIGLA_TIPO.fillna('')+' '+df.NOME_LOGRA.fillna(''), df.NUMERO_IMO)
    l.index = n.index = df.index
    return l, n
lc0, nc = chave_identidade(*separar_logradouro_numero(C.LOGRAD_NUM))
lc0.index = nc.index = C.index
lb, nb = chaves_bhmap(B)
mapa = conciliar_logradouros(lc0.unique(), lb.unique(), limiar=90, margem=2)
lc = lc0.map(lambda v: mapa.get(v, v))
kc, kb = chave(lc, nc), chave(lb, nb)
tipo_i = iptu.TIPO_LOGRADOURO.fillna('').str.replace(r'^BEC$', 'BECO', regex=True)
li0, ni = chave_identidade(tipo_i+' '+iptu.NOME_LOGRADOURO.fillna(''), iptu.NUMERO_IMOVEL)
mapa_i = conciliar_logradouros(li0.unique(), lb.unique(), limiar=90, margem=2)
li = li0.map(lambda v: mapa_i.get(v, v))
ki = chave(li, ni)
SC, SB, SI = set(kc.dropna()), set(kb.dropna()), set(ki.dropna())
comum, so_c, so_b = SC & SB, SC - SB, SB - SC
reg_por_chave = kc.dropna().value_counts()
pts_por_chave = kb.dropna().value_counts()
mult_b = B[kb.isin(pts_por_chave.index[pts_por_chave.gt(1)])].assign(k=kb)
ext = mult_b.groupby('k').agg(bairros=('NOME_BAIRR', 'nunique'), x0=('geometry', lambda g: g.x.min()), x1=('geometry', lambda g: g.x.max()),
                              y0=('geometry', lambda g: g.y.min()), y1=('geometry', lambda g: g.y.max()))
ext['span'] = np.hypot(ext.x1-ext.x0, ext.y1-ext.y0)
condensacao = pd.Series({
    'cnefe_registros': len(C), 'cnefe_registros_com_chave': int(kc.notna().sum()), 'cnefe_registros_sem_numero': int(kc.isna().sum()),
    'cnefe_chaves': len(SC), 'cnefe_registros_por_chave_media': reg_por_chave.mean(), 'cnefe_chaves_um_registro': int(reg_por_chave.eq(1).sum()),
    'cnefe_registros_em_chaves_multiplas': int(reg_por_chave[reg_por_chave.gt(1)].sum()),
    'bhmap_pontos': len(B), 'bhmap_pontos_com_chave': int(kb.notna().sum()), 'bhmap_chaves': len(SB),
    'bhmap_chaves_varios_pontos': int(pts_por_chave.gt(1).sum()), 'bhmap_pontos_em_chaves_varias': int(pts_por_chave[pts_por_chave.gt(1)].sum()),
    'bhmap_chaves_varios_bairros': int(ext.bairros.gt(1).sum()), 'bhmap_chaves_extensao_100m': int(ext.span.gt(100).sum()),
    'vocabulario_bhmap_logradouros': int(pd.Series(lb.dropna().unique()).size), 'cnefe_logradouros_distintos': int(lc0.dropna().nunique()),
    'nomes_cnefe_conciliados': sum(1 for k, v in mapa.items() if k != v),
    'iptu_unidades': len(iptu), 'iptu_lotes': iptu.NULOTCTM.nunique(), 'iptu_chaves': len(SI), 'nomes_iptu_conciliados': sum(1 for k, v in mapa_i.items() if k != v),
    'chaves_comuns': len(comum), 'chaves_so_cnefe': len(so_c), 'chaves_so_bhmap': len(so_b)})
condensacao.to_csv(T/'nb23_condensacao_chaves.csv', header=['n'])
print(condensacao.round(2).to_string())

cnefe_registros                        1180102.00
cnefe_registros_com_chave              1157750.00
cnefe_registros_sem_numero               22352.00
cnefe_chaves                            456022.00
cnefe_registros_por_chave_media              2.54
cnefe_chaves_um_registro                257058.00
cnefe_registros_em_chaves_multiplas     900692.00
bhmap_pontos                            757113.00
bhmap_pontos_com_chave                  757113.00
bhmap_chaves                            613763.00
bhmap_chaves_varios_pontos               95364.00
bhmap_pontos_em_chaves_varias           238714.00
bhmap_chaves_varios_bairros               9531.00
bhmap_chaves_extensao_100m               10701.00
vocabulario_bhmap_logradouros            13275.00
cnefe_logradouros_distintos              12628.00
nomes_cnefe_conciliados                   1335.00
iptu_unidades                           896964.00
iptu_lotes                              306352.00
iptu_chaves                             324661.00


Os 1.157.750 registros do CNEFE com número formam 456.022 chaves, em média 2,54 registros por chave, porque cada unidade de um edifício repete o endereço da porta. Ficam sem chave 22.352 registros sem número. Mais da metade das chaves (257.058) tem um único registro, e as demais reúnem 900.692 registros. No BHMap, os 757.113 pontos formam 613.763 chaves. A maior parte tem um único ponto, mas 95.364 chaves têm vários, somando 238.714 pontos. Em 9.531 delas os pontos estão em mais de um bairro e em 10.701 se espalham por mais de 100 m, sinal de nomes de logradouro repetidos. Os nomes do BHMap são 13.275, e 1.335 nomes do CNEFE foram conciliados a eles. As duas bases têm 404.796 chaves em comum, 51.226 só no CNEFE e 208.967 só no BHMap.

## 3. Pareamento pela chave

Cada registro do CNEFE é ligado a um ponto do BHMap com a mesma chave, sem limite de distância. Quando a chave tem mais de um ponto, fica o mais próximo do registro, o que favorece distâncias curtas sem confirmar a identidade. A chave é marcada como ambígua quando seus pontos se espalham por mais de 100 m. A reconstrução com bairro e CEP de cada lado, usada na Seção 8, precisa reproduzir exatamente as distâncias e os pontos escolhidos.

In [3]:
I = parear_por_identidade(C, B, (lc0, nc), (lb, nb), mapa_logradouros=mapa)
I['COD_UNICO_ENDERECO'] = C.COD_UNICO_ENDERECO.values
assert int(I.pareado_identidade.sum()) == int(kc.isin(comum).sum())
I.to_parquet(P/'cnefe_identidade_completa.parquet')
def cep(s):
    x = s.astype('string[python]').str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True).str.zfill(8)
    return x.where(x.str.fullmatch(r'\d{8}', na=False) & x.ne('00000000'))
def bairro(s):
    return s.map(lambda v: normalize_logradouro(v) if pd.notna(v) else pd.NA).astype('string[python]').replace('', pd.NA)
A = pd.DataFrame({'idx': C.index, 'chave': kc.to_numpy(), 'xc': C.geometry.x.to_numpy(), 'yc': C.geometry.y.to_numpy(),
                  'bairro_c': bairro(C.DSC_LOCALIDADE).to_numpy(), 'cep_c': cep(C.CEP).to_numpy()})
R = pd.DataFrame({'idx_b': B.index, 'chave': kb.to_numpy(), 'xb': B.geometry.x.to_numpy(), 'yb': B.geometry.y.to_numpy(),
                  'bairro_b': bairro(B.NOME_BAIRR).to_numpy(), 'cep_b': cep(B.CEP).to_numpy()})
J = A.dropna(subset=['chave']).merge(R.dropna(subset=['chave']), on='chave', how='inner')
J['dist'] = np.hypot(J.xc-J.xb, J.yc-J.yb)
base = J.loc[J.groupby('idx').dist.idxmin()].set_index('idx').sort_index()
assert len(base) == int(I.pareado_identidade.sum())
assert np.allclose(base.dist, I.dist_identidade.reindex(base.index), atol=1e-8)
assert np.allclose(base.xb, I.ref_x.reindex(base.index), atol=1e-8) and np.allclose(base.yb, I.ref_y.reindex(base.index), atol=1e-8)
d = base.dist
dk = pd.Series(d.to_numpy(), index=base.chave.to_numpy()).groupby(level=0).median()
ncoord = R.dropna(subset=['chave']).assign(x=lambda r: r.xb.round(2), y=lambda r: r.yb.round(2)).drop_duplicates(['chave', 'x', 'y']).groupby('chave').size()
pareamento = pd.Series({
    'registros_pareados': len(base), 'cobertura_pct': 100*len(base)/len(C),
    'registros_chave_ausente_bhmap': int(kc.isin(so_c).sum()), 'registros_sem_numero': int(kc.isna().sum()),
    'mediana_m': d.median(), 'p90_m': d.quantile(.9), 'ate_10m_pct': 100*d.le(10).mean(), 'ate_30m_pct': 100*d.le(30).mean(),
    'acima_100m': int(d.gt(100).sum()), 'acima_100m_pct': 100*d.gt(100).mean(), 'acima_1km': int(d.gt(1000).sum()),
    'registros_chave_varias_coordenadas': int(base.chave.map(ncoord).gt(1).sum()),
    'registros_chave_ambigua': int(I.chave_ambigua.sum()),
    'chaves_pareadas': len(dk), 'mediana_por_chave_m': dk.median(), 'p90_por_chave_m': dk.quantile(.9),
    'cobertura_em_vila_pct': 100*I.pareado_identidade[em_vila].mean(), 'cobertura_fora_vila_pct': 100*I.pareado_identidade[~em_vila].mean(),
    'candidatos': len(J)})
pareamento.to_csv(T/'nb23_pareamento.csv', header=['valor'])
print(pareamento.round(2).to_string())


registros_pareados                    1062044.00
cobertura_pct                              90.00
registros_chave_ausente_bhmap           95706.00
registros_sem_numero                    22352.00
mediana_m                                   9.84
p90_m                                      35.59
ate_10m_pct                                50.84
ate_30m_pct                                87.73
acima_100m                              29775.00
acima_100m_pct                              2.80
acima_1km                                8101.00
registros_chave_varias_coordenadas     243456.00
registros_chave_ambigua                 28962.00
chaves_pareadas                        404796.00
mediana_por_chave_m                         8.36
p90_por_chave_m                            21.60
cobertura_em_vila_pct                      72.33
cobertura_fora_vila_pct                    92.15
candidatos                            1485150.00


Pela chave, 1.062.044 registros encontram par, 90,0% do CNEFE. A mediana é de 9,84 m e o P90 de 35,59 m, com 50,8% dos pares a até 10 m. Ficam a mais de 100 m 29.775 pares (2,80%), e a mais de 1 km 8.101. Têm mais de uma coordenada candidata 243.456 registros pareados, e 28.962 estão em chaves ambíguas. Tomando a mediana dentro de cada uma das 404.796 chaves pareadas, a mediana entre chaves é de 8,36 m e o P90 de 21,60 m. A cobertura é de 72,3% dentro das vilas e favelas e de 92,1% fora delas.


## 4. Sensibilidade à conciliação de nomes

O limiar de conciliação varia do casamento exato até 85, sempre com margem de 2.

In [4]:
linhas = []
pareado_exato = None
for limiar in [None, 95, 92, 90, 88, 85]:
    m = {} if limiar is None else conciliar_logradouros(lc0.unique(), lb.unique(), limiar=limiar, margem=2)
    r = parear_por_identidade(C, B, (lc0, nc), (lb, nb), mapa_logradouros=m)
    if limiar is None: pareado_exato = r.pareado_identidade
    dd = r.dist_identidade[r.pareado_identidade]
    linhas.append({'limiar': 'exato' if limiar is None else limiar, 'nomes_conciliados': sum(1 for k, v in m.items() if k != v),
                   'cobertura_pct': 100*len(dd)/len(C), 'mediana_m': dd.median(), 'p90_m': dd.quantile(.9), 'acima_100m_pct': 100*dd.gt(100).mean(),
                   'em_vila_pct': 100*r.pareado_identidade[em_vila].mean(), 'fora_vila_pct': 100*r.pareado_identidade[~em_vila].mean()})
sens_conc = pd.DataFrame(linhas)
sens_conc.to_csv(T/'nb23_sensibilidade_conciliacao.csv', index=False)
recup = I.pareado_identidade & ~pareado_exato
conc_resumo = pd.Series({'registros_recuperados_pela_conciliacao': int(recup.sum()), 'recuperados_em_vila_pct': 100*em_vila[recup].mean(),
                         'vila_pct_dos_registros': 100*em_vila.mean()})
conc_resumo.to_csv(T/'nb23_conciliacao_recuperados.csv', header=['valor'])
print(sens_conc.round(2).to_string(index=False)); print(conc_resumo.round(2).to_string())

limiar  nomes_conciliados  cobertura_pct  mediana_m  p90_m  acima_100m_pct  em_vila_pct  fora_vila_pct
 exato                  0          86.91       9.80  35.02            2.64        56.67          90.59
    95                963          88.99       9.84  35.53            2.76        68.82          91.45
    92               1204          89.64       9.83  35.50            2.77        70.63          91.96
    90               1335          90.00       9.84  35.59            2.80        72.33          92.15
    88               1430          90.14       9.84  35.63            2.82        72.74          92.26
    85               1543          90.29       9.84  35.66            2.84        73.19          92.37
registros_recuperados_pela_conciliacao    36414.00
recuperados_em_vila_pct                      55.11
vila_pct_dos_registros                       10.86


A mediana fica entre 9,80 m e 9,84 m e o P90 entre 35,0 m e 35,7 m em toda a faixa. A cobertura muda, e de forma desigual. Nas vilas e favelas vai de 56,7% no casamento exato para 72,3% no limiar 90, e fora delas de 90,6% para 92,2%. A conciliação recupera 36.414 registros, dos quais 55,1% estão nas vilas e favelas, que têm 10,9% dos registros.

## 5. Chaves exclusivas e evidência fiscal

As chaves de cada lado são cruzadas com o IPTU. As chaves que só uma base tem recebem a primeira classificação aplicável: outro número no mesmo lote e logradouro na outra base, logradouro presente sem aquele número, ou logradouro ausente da outra base mesmo após a conciliação. Cada ponto é ligado ao lote mais próximo a até 10 m. Para os registros do CNEFE sem par, o apoio fiscal é estar a até 10 m de um lote que o IPTU associa à mesma chave.

In [5]:
regioes = []
for c_ in [True, False]:
    for b_ in [True, False]:
        for i_ in [True, False]:
            if not (c_ or b_ or i_): continue
            s = (SC if c_ else (SB|SI)-SC) & (SB if b_ else (SC|SI)-SB) & (SI if i_ else (SC|SB)-SI)
            regioes.append({'cnefe': c_, 'bhmap': b_, 'iptu': i_, 'chaves': len(s)})
regioes = pd.DataFrame(regioes)
assert regioes.chaves.sum() == len(SC|SB|SI)
regioes.to_csv(T/'nb23_conjuntos_tres_cadastros.csv', index=False)
def lote_proximo(pts):
    j = gpd.sjoin_nearest(pts[['geometry']], lotes.reset_index()[['NULOTCTM', 'geometry']], how='left', max_distance=TOL)
    return j[~j.index.duplicated()].NULOTCTM
lote_b = lote_proximo(B)
lote_c = lote_proximo(C[kc.notna()])
def mesmo_lote_outro_numero(lote_alvo, log_alvo, num_alvo, chave_alvo, lote_outro, log_outro, num_outro):
    a = pd.DataFrame({'lote': lote_alvo, 'log': log_alvo.reindex(lote_alvo.index), 'num': num_alvo.reindex(lote_alvo.index),
                      'chave': chave_alvo.reindex(lote_alvo.index)}).dropna()
    o = pd.DataFrame({'lote': lote_outro, 'log': log_outro.reindex(lote_outro.index), 'num_o': num_outro.reindex(lote_outro.index)}).dropna().drop_duplicates()
    j = a.merge(o, on=['lote', 'log'])
    return set(j.loc[j.num.ne(j.num_o), 'chave'])
def classificar(exclusivas, logs_outro, ml):
    rot = pd.Series('rua ausente da outra base', index=sorted(exclusivas))
    rot[pd.Series([k.split('|')[0] for k in rot.index], index=rot.index).isin(logs_outro)] = 'rua presente, número ausente'
    rot[rot.index.isin(list(ml))] = 'mesmo lote, outro número'
    return rot
m_cx = kc.isin(so_c); m_bx = kb.isin(so_b)
classe_c = classificar(so_c, set(lb.dropna()), mesmo_lote_outro_numero(lote_c[lote_c.index.isin(C.index[m_cx])], lc, nc, kc, lote_b, lb, nb))
classe_b = classificar(so_b, set(lc.dropna()), mesmo_lote_outro_numero(lote_b[m_bx], lb, nb, kb, lote_c, lc, nc))
def resumo_classes(classe, chaves_reg, vila_reg, nome):
    reg = pd.DataFrame({'chave': chaves_reg, 'vila': vila_reg})
    reg = reg[reg.chave.isin(classe.index)]
    reg['classe'] = reg.chave.map(classe)
    t = pd.DataFrame({'classe': classe, 'iptu': classe.index.isin(list(SI))})
    r = t.groupby('classe').agg(chaves=('iptu', 'size'), chaves_no_iptu=('iptu', 'sum'))
    r['registros'] = reg.groupby('classe').size(); r['registros_em_vila'] = reg.groupby('classe').vila.sum()
    r.loc['total'] = r.sum()
    r['pct_chaves'] = 100*r.chaves/len(classe); r['pct_no_iptu'] = 100*r.chaves_no_iptu/r.chaves
    r['pct_registros_em_vila'] = 100*r.registros_em_vila/r.registros
    return r.assign(lado=nome)
cls = pd.concat([resumo_classes(classe_c, kc, em_vila, 'só CNEFE'), resumo_classes(classe_b, kb, em_vila_b, 'só BHMap')])
cls.to_csv(T/'nb23_classes_exclusivas.csv')
print(regioes.to_string(index=False)); print(cls.round(1).to_string())


 cnefe  bhmap  iptu  chaves
  True   True  True  271318
  True   True False  133478
  True  False  True    1171
  True  False False   50055
 False   True  True   48341
 False   True False  160626
 False  False  True    3831
                              chaves  chaves_no_iptu  registros  registros_em_vila  pct_chaves  pct_no_iptu  pct_registros_em_vila      lado
classe                                                                                                                                      
mesmo lote, outro número       18802             665      34224               9288        36.7          3.5                   27.1  só CNEFE
rua ausente da outra base      16493             304      33197              11022        32.2          1.8                   33.2  só CNEFE
rua presente, número ausente   15931             202      28285              10123        31.1          1.3                   35.8  só CNEFE
total                          51226            1171      95706        

In [6]:
kl = pd.DataFrame({'chave': ki, 'NULOTCTM': iptu.NULOTCTM, 'logr': li}).dropna(subset=['chave']).drop_duplicates(['chave', 'NULOTCTM'])
kl = kl[kl.NULOTCTM.isin(lotes.index)].copy()
E = pd.DataFrame({'chave': kc[m_cx], 'x': C.geometry.x[m_cx], 'y': C.geometry.y[m_cx]}).reset_index().merge(kl[['chave', 'NULOTCTM']], on='chave')
E['d'] = shapely.distance(shapely.points(E.x.to_numpy(), E.y.to_numpy()), lotes.geometry.reindex(E.NULOTCTM).to_numpy())
apoio = E.groupby('index').d.min().reindex(C.index).le(TOL) & m_cx
apoio_tab = pd.Series({'registros_so_cnefe': int(m_cx.sum()), 'com_apoio_fiscal_10m': int(apoio.sum()),
                       'pct_apoio': 100*apoio.sum()/m_cx.sum(), 'registros_so_cnefe_em_vila_pct': 100*em_vila[m_cx].mean(),
                       'apoio_em_vila': int((apoio & em_vila).sum())})
apoio_tab.to_csv(T/'nb23_apoio_fiscal_so_cnefe.csv', header=['valor'])
print(apoio_tab.round(2).to_string())

registros_so_cnefe                95706.00
com_apoio_fiscal_10m               3082.00
pct_apoio                             3.22
registros_so_cnefe_em_vila_pct       31.80
apoio_em_vila                        26.00


Das 51.226 chaves só do CNEFE, 36,7% têm outro número no mesmo lote do BHMap, 31,1% estão em logradouros do BHMap sem aquele número e 32,2% em logradouros ausentes dele. Só 2,3% estão no IPTU, e 31,8% dos seus 95.706 registros estão nas vilas e favelas, quase três vezes a participação desses territórios no CNEFE (10,9%). Só 3.082 desses registros (3,2%) têm apoio fiscal.

Das 208.967 chaves só do BHMap, 54,3% têm um registro do CNEFE no mesmo lote e logradouro com outro número, 33,7% estão em logradouros do CNEFE sem aquele número e 12,1% em logradouros ausentes do CNEFE. Estão no IPTU 23,1% delas.


## 6. Evidência para os pares a mais de 100 m

O controle repete o do NB16: nos pares já próximos, mede-se quantas vezes cada ponto está junto ao lote que o IPTU associa à chave, com tolerâncias de 0 a 20 m. As classes da cauda seguem a mesma ordem de precedência e os mesmos parâmetros do NB16.

A classe de lote extenso admite o mesmo lote mais próximo dos dois pontos ou lotes cujos centroides têm retângulo envolvente de diagonal de até 100 m. As distâncias às vias são medidas até os componentes formados pelos buffers de 25 m. Sem geometria da via, sua distância é desconhecida e não sustenta a classe CNEFE fora da via nomeada.

In [7]:
centro = lotes.geometry.centroid
kl['cx'] = centro.reindex(kl.NULOTCTM).x.to_numpy(); kl['cy'] = centro.reindex(kl.NULOTCTM).y.to_numpy()
g = kl.groupby('chave')
disp_iptu = np.hypot(g.cx.max()-g.cx.min(), g.cy.max()-g.cy.min()).rename('dispersao_iptu_m')
par = I.pareado_identidade
D = pd.DataFrame({'chave': kc[par], 'dist': I.loc[par, 'dist_identidade'], 'xc': C.geometry.x[par], 'yc': C.geometry.y[par],
                  'xb': I.loc[par, 'ref_x'], 'yb': I.loc[par, 'ref_y'], 'conciliada': lc0[par].ne(lc[par]),
                  'ambigua': I.loc[par, 'chave_ambigua'].astype(bool), 'NV_GEO_COORD': C.loc[par, 'NV_GEO_COORD']})
D = D.join(disp_iptu, on='chave')
D['chave_no_iptu'] = D.chave.isin(set(kl.chave))
E2 = D[['chave', 'xc', 'yc', 'xb', 'yb']].reset_index().merge(kl[['chave', 'NULOTCTM']], on='chave')
geo = lotes.geometry.reindex(E2.NULOTCTM).to_numpy()
E2['dC'] = shapely.distance(shapely.points(E2.xc.to_numpy(), E2.yc.to_numpy()), geo)
E2['dB'] = shapely.distance(shapely.points(E2.xb.to_numpy(), E2.yb.to_numpy()), geo)
iC = E2.loc[E2.groupby('index').dC.idxmin(), ['index', 'NULOTCTM', 'dC']].set_index('index').rename(columns={'NULOTCTM': 'lote_chave_c'})
iB = E2.loc[E2.groupby('index').dB.idxmin(), ['index', 'NULOTCTM', 'dB']].set_index('index').rename(columns={'NULOTCTM': 'lote_chave_b'})
D = D.join(iC).join(iB)
D['faixa'] = pd.cut(D.dist, [-np.inf, 10, 100, np.inf], labels=['até 10 m', '10 a 100 m', 'acima de 100 m'])
linhas = []
for tol in [0, 5, 10, 20]:
    for faixa, s in D.groupby('faixa', observed=True):
        e = s[s.chave_no_iptu]
        linhas.append({'tolerancia_m': tol, 'faixa': faixa, 'pares': len(s), 'pares_com_chave_no_iptu': len(e),
                       'pct_cnefe_no_lote': 100*(e.dC <= tol).mean(), 'pct_bhmap_no_lote': 100*(e.dB <= tol).mean(),
                       'pct_ambos': 100*((e.dC <= tol) & (e.dB <= tol)).mean(), 'pct_nenhum': 100*((e.dC > tol) & (e.dB > tol)).mean()})
controle = pd.DataFrame(linhas)
controle.to_csv(T/'nb23_controle_evidencia.csv', index=False)
print(controle.round(2).to_string(index=False))

 tolerancia_m          faixa  pares  pares_com_chave_no_iptu  pct_cnefe_no_lote  pct_bhmap_no_lote  pct_ambos  pct_nenhum
            0       até 10 m 539913                   419749              20.30              98.44      20.25        1.51
            0     10 a 100 m 492356                   383134              22.36              98.43      22.25        1.47
            0 acima de 100 m  29775                    18979              48.64              94.58      48.35        5.13
            5       até 10 m 539913                   419749              73.43              98.96      73.32        0.93
            5     10 a 100 m 492356                   383134              34.25              98.85      34.07        0.97
            5 acima de 100 m  29775                    18979              50.30              94.78      49.90        4.82
           10       até 10 m 539913                   419749              99.22              99.30      99.18        0.66
           10     10 a 1

Nos pares a até 10 m com a chave no IPTU, os dois pontos estão junto ao lote da chave em 99,2% dos casos com 10 m de tolerância, e o ponto do BHMap está dentro do polígono em 98,4% com tolerância zero.

In [8]:
via = gpd.read_parquet(P/'declividade.parquet', columns=['NOME_LOGRA', 'geometry'])
lv, _ = chave_identidade(via.NOME_LOGRA.fillna(''), pd.Series(['0']*len(via)))
mapa_v = conciliar_logradouros(lv.unique(), lb.unique(), limiar=90, margem=2)
via['logr'] = lv.map(lambda v: mapa_v.get(v, v)).to_numpy()
comp = []
for nome, gv in via.groupby('logr'):
    u = shapely.union_all(shapely.buffer(gv.geometry.to_numpy(), 25))
    partes = list(getattr(u, 'geoms', [u]))
    comp += [(nome, i, len(partes), p) for i, p in enumerate(partes)]
comp = gpd.GeoDataFrame(comp, columns=['logr', 'componente', 'n_componentes', 'geometry'], crs=via.crs)
def adjudicar(D, tol=TOL, via_m=15):
    s = D.copy()
    s['logr'] = s.chave.str.split('|').str[0]
    E = s[['logr', 'xc', 'yc', 'xb', 'yb']].reset_index().merge(comp, on='logr')
    gg = E.geometry.to_numpy()
    E['dvc'] = shapely.distance(shapely.points(E.xc.to_numpy(), E.yc.to_numpy()), gg)
    E['dvb'] = shapely.distance(shapely.points(E.xb.to_numpy(), E.yb.to_numpy()), gg)
    vc = E.loc[E.groupby('index').dvc.idxmin(), ['index', 'componente', 'dvc']].set_index('index').rename(columns={'componente': 'comp_c'})
    vb = E.loc[E.groupby('index').dvb.idxmin(), ['index', 'componente', 'dvb']].set_index('index').rename(columns={'componente': 'comp_b'})
    s = s.join(vc).join(vb)
    c_lote, b_lote = s.dC <= tol, s.dB <= tol
    c_via, b_via = s.dvc <= via_m, s.dvb <= via_m
    disp = s.dispersao_iptu_m.fillna(0)
    mesmo_lote = c_lote & b_lote & (s.lote_chave_c.eq(s.lote_chave_b) | (disp <= 100))
    homonimia = (c_lote & b_lote & ~mesmo_lote) | s.ambigua | (c_via & b_via & s.comp_c.ne(s.comp_b))
    ev = s.chave_no_iptu
    s['veredito'] = np.select([mesmo_lote, homonimia, ~c_lote & b_lote, c_lote & ~b_lote, ~ev & s.conciliada, ~ev & c_via & b_via, ~ev & s.dvc.notna() & ~c_via],
        ['imovel extenso, mesmo lote', 'homonimia, chave em dois lugares', 'CNEFE fora do lote fiscal', 'BHMap fora do lote fiscal',
         'grafia conciliada sem confirmacao', 'mesma via sem lote fiscal', 'CNEFE fora da via nomeada'], 'indeterminado')
    natureza = {'imovel extenso, mesmo lote': 'nenhuma das duas', 'homonimia, chave em dois lugares': 'casamento textual',
                'grafia conciliada sem confirmacao': 'casamento textual', 'CNEFE fora do lote fiscal': 'coordenada do CNEFE',
                'CNEFE fora da via nomeada': 'coordenada ou endereco do CNEFE', 'BHMap fora do lote fiscal': 'referencia BHMap',
                'mesma via sem lote fiscal': 'posicao ao longo da via', 'indeterminado': 'indeterminado'}
    s['natureza'] = s.veredito.map(natureza)
    return s
cauda = adjudicar(D[D.dist > 100])
assert cauda.loc[cauda.veredito.eq('CNEFE fora da via nomeada'), 'dvc'].notna().all()
assert len(cauda) == 29775
tab = cauda.groupby(['natureza', 'veredito']).dist.agg(pares='size', mediana_m='median', p90_m=lambda x: x.quantile(.9)).reset_index()
tab['pct_cauda'] = 100*tab.pares/len(cauda)
tab = tab.sort_values('pares', ascending=False)
tab.to_csv(T/'nb23_veredito_cauda.csv', index=False)
sens = []
for tol in [5, 10, 20]:
    for via_m in [5, 15, 25]:
        s = adjudicar(D[D.dist > 100], tol=tol, via_m=via_m)
        for k, v in s.veredito.value_counts().items():
            sens.append({'tolerancia_lote_m': tol, 'distancia_via_m': via_m+25, 'veredito': k, 'pares': v, 'pct': 100*v/len(s)})
sens = pd.DataFrame(sens)
sens.to_csv(T/'nb23_sensibilidade_tolerancia.csv', index=False)
cauda[['chave', 'dist', 'xc', 'yc', 'xb', 'yb', 'conciliada', 'ambigua', 'NV_GEO_COORD', 'chave_no_iptu',
       'dispersao_iptu_m', 'dC', 'dB', 'dvc', 'dvb', 'veredito', 'natureza']].rename_axis('idx_cnefe_original').to_parquet(T/'nb23_cauda_adjudicada.parquet')
print('pares na cauda', len(cauda)); print(tab.round(1).to_string(index=False))
print(sens.groupby('veredito').pct.agg(['min', 'max']).round(1).to_string())


pares na cauda 29775
                       natureza                          veredito  pares  mediana_m   p90_m  pct_cauda
               nenhuma das duas        imovel extenso, mesmo lote   9521      129.3   191.4       32.0
            coordenada do CNEFE         CNEFE fora do lote fiscal   7662      171.3   955.9       25.7
              casamento textual  homonimia, chave em dois lugares   5515     4744.6 15749.9       18.5
coordenada ou endereco do CNEFE         CNEFE fora da via nomeada   3754     2225.1 18211.1       12.6
        posicao ao longo da via         mesma via sem lote fiscal   2162      162.2   516.8        7.3
              casamento textual grafia conciliada sem confirmacao    914     4545.7 14482.8        3.1
                  indeterminado                     indeterminado    156      361.3 15964.6        0.5
               referencia BHMap         BHMap fora do lote fiscal     91      228.2   404.3        0.3
                                    min   max
veredi

Dos 29.775 pares a mais de 100 m, 9.521 (32,0%) atendem à classe de imóvel extenso, com os dois pontos perto do mesmo lote ligado ou de lotes cujos centroides atendem ao critério de dispersão de até 100 m. A mediana dessa classe é de 129 m. As chaves ambíguas somam 5.515 (18,5%), com mediana de 4,7 km, e a grafia conciliada sem confirmação 914 (3,1%). O ponto do CNEFE está longe do lote ligado próximo ao ponto do BHMap em 7.662 pares (25,7%), com mediana de 171 m. Sem a chave no IPTU, 3.754 pares (12,6%) têm o ponto do CNEFE fora da via nomeada e 2.162 (7,3%) seguem a mesma via. Há 91 pares na classe BHMap fora do lote e 156 indeterminados. A ausência de geometria do logradouro não demonstra que a coordenada está fora dele. As parcelas variam pouco entre as nove combinações de tolerância testadas.

## 7. Geometria de referência e apartamentos

Comparamos a distância ponto a ponto com a distância do CNEFE ao lote ligado à chave, usando apenas chaves com um único lote no IPTU. A comparação mede a sensibilidade à geometria de referência, sem isolar a contribuição causal da posição do ponto municipal dentro do lote. O contraste entre apartamentos e os demais registros que não são estabelecimentos usa intervalos percentis de 95% com 500 reamostragens de blocos espaciais de 2 km. Os quintis vêm da área do lote nos registros dos dois grupos reunidos. A diferença entre as medianas em cada quintil é ponderada pela participação desse quintil no total de apartamentos, mantendo limites e pesos fixos nas reamostragens.

In [9]:
n_lotes_chave = kl.groupby('chave').NULOTCTM.nunique()
chaves_um_lote = set(n_lotes_chave[n_lotes_chave.eq(1)].index)
R8 = D[D.chave.isin(chaves_um_lote)].join(pd.DataFrame({'apartamento': apto, 'estabelecimento': estab, 'em_vila': em_vila}))
grupos = {'todos': R8.index == R8.index, 'apartamentos': R8.apartamento, 'demais nao estabelecimentos': ~R8.apartamento & ~R8.estabelecimento,
          'estabelecimentos': R8.estabelecimento, 'em vila': R8.em_vila, 'fora de vila': ~R8.em_vila}
def estat(s, rotulo):
    return {f'{rotulo}_mediana_m': s.median(), f'{rotulo}_p90_m': s.quantile(.9), f'{rotulo}_acima30_pct': 100*s.gt(30).mean()}
pl = pd.DataFrame([{'grupo': g_, 'pares': int(m.sum()), **estat(R8.dist[m], 'ponto_ponto'), **estat(R8.dC[m], 'ponto_lote'),
                    'cnefe_dentro_do_lote_pct': 100*R8.dC[m].eq(0).mean()} for g_, m in grupos.items()])
pl.to_csv(T/'nb23_ponto_lote.csv', index=False)
R8['bloco'] = (R8.xc//2000).astype(int).astype(str)+'_'+(R8.yc//2000).astype(int).astype(str)
ap_, ca_ = R8.apartamento.to_numpy(), (~R8.apartamento & ~R8.estabelecimento).to_numpy()
valores = R8[['dist', 'dC']].to_numpy()
pos = {b: np.asarray(ix) for b, ix in R8.groupby('bloco').indices.items()}; blocos = list(pos)
rng = np.random.default_rng(SEED)
reps = []
for _ in range(500):
    i = np.concatenate([pos[blocos[k]] for k in rng.integers(len(blocos), size=len(blocos))])
    v, a, c_ = valores[i], ap_[i], ca_[i]
    reps.append([np.median(v[a, 0])-np.median(v[c_, 0]), np.median(v[a, 1])-np.median(v[c_, 1])])
reps = np.array(reps)
contraste = pd.DataFrame({'medida': ['ponto_ponto', 'ponto_lote'],
                          'diferenca_m': [np.median(valores[ap_, 0])-np.median(valores[ca_, 0]), np.median(valores[ap_, 1])-np.median(valores[ca_, 1])],
                          'ic95_inf': np.percentile(reps, 2.5, axis=0), 'ic95_sup': np.percentile(reps, 97.5, axis=0)})
R8['area_lote'] = lotes.AREA_M2.reindex(R8.lote_chave_c).to_numpy()
base_ac = R8[(R8.apartamento | (~R8.apartamento & ~R8.estabelecimento)) & R8.area_lote.notna()].copy()
base_ac['quintil_area'] = pd.qcut(base_ac.area_lote, 5, labels=False)
q_, a_, v_ = base_ac.quintil_area.to_numpy(), base_ac.apartamento.to_numpy(), base_ac[['dist', 'dC']].to_numpy()
pesos = np.bincount(q_[a_], minlength=5)/a_.sum()
def estratificado(v, q, a):
    out = np.zeros(2)
    for k in range(5):
        mk = q == k
        if (mk & a).any() and (mk & ~a).any():
            out += pesos[k]*(np.median(v[mk & a], axis=0)-np.median(v[mk & ~a], axis=0))
    return out
pos_ac = {b: np.asarray(ix) for b, ix in base_ac.reset_index(drop=True).groupby(base_ac.bloco.to_numpy()).indices.items()}
bl_ac = list(pos_ac); reps_e = []
for _ in range(500):
    i = np.concatenate([pos_ac[bl_ac[k]] for k in rng.integers(len(bl_ac), size=len(bl_ac))])
    reps_e.append(estratificado(v_[i], q_[i], a_[i]))
reps_e = np.array(reps_e); est = estratificado(v_, q_, a_)
contraste = pd.concat([contraste, pd.DataFrame({'medida': ['ponto_ponto_estratificado', 'ponto_lote_estratificado'], 'diferenca_m': est,
                       'ic95_inf': np.percentile(reps_e, 2.5, axis=0), 'ic95_sup': np.percentile(reps_e, 97.5, axis=0)})])
contraste.to_csv(T/'nb23_ponto_lote_contraste.csv', index=False)
quintil_maior = pd.Series({'apartamentos_no_maior_quintil_pct': 100*(q_[a_] == 4).mean(), 'demais_no_maior_quintil_pct': 100*(q_[~a_] == 4).mean()})
quintil_maior.to_csv(T/'nb23_quintil_area.csv', header=['valor'])
R8[['dist', 'dC']].rename_axis('idx_cnefe_original').to_parquet(T/'nb23_ponto_lote_pares.parquet')
print('pares com lote único:', len(R8)); print(pl.round(2).to_string(index=False)); print(contraste.round(2).to_string(index=False)); print(quintil_maior.round(1).to_string())

pares com lote único: 808059
                      grupo  pares  ponto_ponto_mediana_m  ponto_ponto_p90_m  ponto_ponto_acima30_pct  ponto_lote_mediana_m  ponto_lote_p90_m  ponto_lote_acima30_pct  cnefe_dentro_do_lote_pct
                      todos 808059                   9.78              35.34                    12.28                  4.34             16.93                    4.40                     21.79
               apartamentos 363643                  13.88              54.51                    21.35                  4.28             20.23                    5.30                     29.46
demais nao estabelecimentos 367131                   7.46              18.27                     4.23                  4.12             12.89                    3.33                     16.08
           estabelecimentos  77285                   9.80              26.20                     7.83                  5.86             20.12                    5.29                     12.75
           

Nos 808.059 pares com um único lote, a mediana ponto a ponto é de 9,78 m e a ponto a lote de 4,34 m, metade. O ponto do CNEFE está dentro do lote em 21,8% dos pares. Na medida ponto a ponto, os apartamentos ficam 6,42 m mais longe que os demais registros (intervalo de 5,37 a 7,50 m). Até o lote, a diferença bruta cai para 0,16 m, com intervalo que inclui zero. Os apartamentos ocupam lotes maiores: 37,5% estão no quinto de maior área, contra 2,6% dos demais. Após estratificar por área do lote, o contraste ponto a ponto é de 4,81 m (3,56 a 6,01 m), e o ponto a lote é de 0,98 m (0,49 a 1,49 m). A comparação mostra sensibilidade à geometria, sem estabelecer a posição do ponto municipal como explicação principal.

## 8. Restrição territorial

Bairro e CEP não entram na chave. A concordância entre eles é medida por faixa de distância e por classe da cauda, e depois o candidato é escolhido de novo exigindo bairro, CEP ou ambos. Como o conjunto restrito está contido no original, nenhum registro mantido pode ficar mais perto, e o código confere a identidade contábil da cauda. Por fim, o recorte comparável ao de Salles et al. (2025) toma os pares da cauda com bairro e CEP iguais, coordenada original e distância acima de 150 m.

In [10]:
base['bairro_disponivel'] = base.bairro_c.notna() & base.bairro_b.notna()
base['cep_disponivel'] = base.cep_c.notna() & base.cep_b.notna()
pares_bairros = base.loc[base.bairro_disponivel, ['bairro_c', 'bairro_b']].drop_duplicates()
semelhante = {(a, b): fuzz.token_sort_ratio(a, b) >= 90 for a, b in pares_bairros.itertuples(index=False, name=None)}
base['mesmo_bairro'] = [semelhante.get((a, b), np.nan) for a, b in zip(base.bairro_c, base.bairro_b)]
base['mesmo_cep'] = base.cep_c.eq(base.cep_b).where(base.cep_disponivel)
base['faixa'] = pd.cut(base.dist, [-1, 10, 30, 100, 1000, np.inf], labels=['até 10 m', '10 a 30 m', '30 a 100 m', '100 m a 1 km', 'acima de 1 km'])
def resumo_texto(g_):
    return pd.Series({'pares': len(g_), 'n_bairro': int(g_.bairro_disponivel.sum()), 'mesmo_bairro_pct': 100*g_.mesmo_bairro.mean(),
                      'n_cep': int(g_.cep_disponivel.sum()), 'mesmo_cep_pct': 100*g_.mesmo_cep.mean()})
por_faixa = base.groupby('faixa', observed=True).apply(resumo_texto, include_groups=False)
por_classe = base.join(cauda[['veredito']]).dropna(subset=['veredito']).groupby('veredito').apply(resumo_texto, include_groups=False)
por_faixa.to_csv(T/'nb23_texto_por_faixa.csv'); por_classe.to_csv(T/'nb23_texto_por_classe.csv')
m_bairro = (J.bairro_c.notna() & J.bairro_b.notna() & J.bairro_c.eq(J.bairro_b)).fillna(False)
m_cep = (J.cep_c.notna() & J.cep_b.notna() & J.cep_c.eq(J.cep_b)).fillna(False)
regras = {'logradouro_numero': np.ones(len(J), dtype=bool), 'mais_bairro': m_bairro, 'mais_cep': m_cep, 'mais_bairro_e_cep': m_bairro & m_cep}
selecionados, linhas = {}, []
for nome, mascara in regras.items():
    cand = J.loc[mascara]
    melhor = cand.loc[cand.groupby('idx').dist.idxmin()].set_index('idx').sort_index()
    selecionados[nome] = melhor
    antes = base.dist.reindex(melhor.index)
    delta = melhor.dist-antes
    assert delta.ge(-1e-8).all()
    linha = {'regra': nome, 'pares': len(melhor), 'cobertura_pct': 100*len(melhor)/len(C), 'mediana_m': melhor.dist.median(), 'p90_m': melhor.dist.quantile(.9),
             'candidato_alterado': int(melhor.idx_b.ne(base.idx_b.reindex(melhor.index)).sum()), 'distancia_aumentou': int(delta.gt(1e-8).sum())}
    for lim in [100, 1000]:
        origem = int(base.dist.gt(lim).sum())
        excluidos = int(base.loc[~base.index.isin(melhor.index), 'dist'].gt(lim).sum())
        entradas = int((antes.le(lim) & melhor.dist.gt(lim)).sum())
        final = int(melhor.dist.gt(lim).sum())
        assert final == origem-excluidos+entradas
        linha.update({f'cauda{lim}_original': origem, f'cauda{lim}_excluida': excluidos, f'cauda{lim}_entrada': entradas, f'cauda{lim}_final': final})
    linhas.append(linha)
restricao = pd.DataFrame(linhas)
restricao.to_csv(T/'nb23_restricao_territorial.csv', index=False)
segmentos = {'todos': pd.Series(True, index=C.index), 'apartamentos': apto, 'demais_tipos': ~apto & ~estab, 'estabelecimentos': estab,
             'em_vila': em_vila, 'fora_vila': ~em_vila}
linhas = []
for nome, melhor in selecionados.items():
    for seg_, m in segmentos.items():
        ids = C.index[m]; pareados = melhor.index.intersection(ids); originais = base.index.intersection(ids)
        linhas.append({'regra': nome, 'segmento': seg_, 'registros': len(ids), 'pareados': len(pareados),
                       'cobertura_pct': 100*len(pareados)/len(ids), 'perda_pp': 100*(len(originais)-len(pareados))/len(ids)})
cobertura = pd.DataFrame(linhas)
cobertura.to_csv(T/'nb23_cobertura_segmentos.csv', index=False)
eleg = pd.DataFrame([{'segmento': s_, 'pareados': len(base.index.intersection(C.index[m])), 'com_lote_unico': len(R8.index.intersection(C.index[m]))}
                     for s_, m in segmentos.items()])
eleg['com_lote_unico_pct'] = 100*eleg.com_lote_unico/eleg.pareados
eleg.to_csv(T/'nb23_elegibilidade_lote.csv', index=False)
cs = cauda.join(base[['mesmo_bairro', 'mesmo_cep']]).join(C.NV_GEO_COORD.rename('nivel'))
recorte = cs[cs.mesmo_bairro.eq(True) & cs.mesmo_cep.eq(True) & cs.nivel.eq(1) & cs.dist.gt(150)]
salles = recorte.groupby('veredito').size().rename('pares').reset_index()
salles['pct'] = 100*salles.pares/len(recorte)
salles.sort_values('pares', ascending=False).to_csv(T/'nb23_recorte_salles.csv', index=False)
print(por_faixa.round(1).to_string()); print(por_classe.round(1).to_string())
print(restricao.round(2).to_string(index=False))
print(cobertura.pivot(index='segmento', columns='regra', values='cobertura_pct')[list(regras)].round(1).to_string())
print(eleg.round(1).to_string(index=False)); print('recorte Salles', len(recorte)); print(salles.round(1).to_string(index=False))

                  pares  n_bairro  mesmo_bairro_pct     n_cep  mesmo_cep_pct
faixa                                                                       
até 10 m       539913.0  539913.0              92.7  538954.0           93.7
10 a 30 m      391803.0  391803.0              92.7  390876.0           91.2
30 a 100 m     100553.0  100553.0              91.3  100360.0           88.8
100 m a 1 km    21674.0   21674.0              82.3   21612.0           88.2
acima de 1 km    8101.0    8101.0               1.0    7705.0            7.1
                                    pares  n_bairro  mesmo_bairro_pct   n_cep  mesmo_cep_pct
veredito                                                                                    
BHMap fora do lote fiscal            91.0      91.0              57.1    91.0           95.6
CNEFE fora da via nomeada          3754.0    3754.0              24.3  3695.0           45.7
CNEFE fora do lote fiscal          7662.0    7662.0              84.6  7661.0           8

Bairro e CEP coincidem em mais de 92% dos pares a até 10 m e quase nunca acima de 1 km (1,0% e 7,1%). Na cauda, a concordância é alta nos imóveis extensos (87,0% e 90,2%) e nos pontos do CNEFE fora do lote (84,6% e 84,2%), e baixa nas chaves ambíguas (8,1% e 13,8%).

Exigir bairro reduz os pares a mais de 1 km de 8.101 para 81, todos por exclusão. Os 462 registros que mudam de candidato ficam mais longe. A cobertura cai de 72,3% para 52,4% nas vilas e favelas e de 92,1% para 86,0% fora delas, uma perda de 19,9 pontos contra 6,1. Com bairro e CEP, só 42,7% dos registros das vilas ficam com par. Tem um único lote vinculado 12,7% dos pares das vilas, contra 82,1% fora delas. No recorte comparável a Salles et al., 5.020 pares, os pontos do CNEFE fora do lote são 52,5% e os imóveis extensos 22,0%.

## 9. Números por segmento

A tabela abre os registros do CNEFE por segmento, com par, sem número, só no CNEFE, com apoio fiscal e a mais de 100 m, e divide a cauda de cada segmento pelas classes da Seção 6.

In [11]:
nat_por_reg = cauda.veredito.reindex(C.index)
linhas = []
for nome, m in segmentos.items():
    p_ = I.pareado_identidade & m
    linhas.append({'segmento': nome, 'registros': int(m.sum()), 'pareados': int(p_.sum()), 'sem_numero': int((kc.isna() & m).sum()),
                   'so_cnefe': int((m_cx & m).sum()), 'so_cnefe_com_apoio_fiscal': int((apoio & m).sum()),
                   'acima_100m': int((p_ & I.dist_identidade.gt(100)).sum()),
                   **{f'cauda_{v}': int((nat_por_reg == v)[m].sum()) for v in tab.veredito}})
seg_tab = pd.DataFrame(linhas)
assert (seg_tab.registros == seg_tab.pareados+seg_tab.sem_numero+seg_tab.so_cnefe).all()
seg_tab.to_csv(T/'nb23_segmentos.csv', index=False)
print(seg_tab.T.to_string())

                                               0             1             2                 3        4          5
segmento                                   todos  apartamentos  demais_tipos  estabelecimentos  em_vila  fora_vila
registros                                1180102        403623        642844            133635   128149    1051953
pareados                                 1062044        385346        561288            115410    92694     969350
sem_numero                                 22352           649         15107              6596     5022      17330
so_cnefe                                   95706         17628         66449             11629    30433      65273
so_cnefe_com_apoio_fiscal                   3082          1651          1206               225       26       3056
acima_100m                                 29775         14523         13300              1952     5977      23798
cauda_imovel extenso, mesmo lote            9521          9308           172    

Os apartamentos têm a maior cobertura (95,5%), seguidos dos demais tipos (87,3%) e dos estabelecimentos (86,4%). Nas vilas e favelas, 6,4% dos pares passam de 100 m, contra 2,5% fora delas, e chaves ambíguas e grafias sem confirmação são 56% dessa cauda. Nos apartamentos, imóveis extensos são 64% da cauda.

## 10. Registro das entradas

O manifesto grava os hashes das entradas e os parâmetros usados.

In [12]:
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
entradas = [P/'cnefe_bh.parquet', P/'bhmap_bh.parquet', P/'lote_ctm.parquet', P/'vila_favela.parquet', P/'declividade.parquet', arquivo_iptu]
manifesto = {'notebook': '23_referencia_municipal_completa.ipynb', 'entradas_sha256': {str(p.relative_to(ROOT)): sha(p) for p in entradas},
             'referencia': 'todos os pontos de endereço do BHMap', 'parametros': {'conciliacao': 'limiar 90, margem 2', 'tolerancia_lote_m': TOL,
             'via_buffer_m': 25, 'via_distancia_m': 40, 'dispersao_ambigua_m': 100}, 'pares': int(len(base)), 'pares_cauda': int(len(cauda))}
(T/'nb23_manifesto.json').write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifesto, ensure_ascii=False, indent=1))

{
 "notebook": "23_referencia_municipal_completa.ipynb",
 "entradas_sha256": {
  "data\\processed\\cnefe_bh.parquet": "b5f69c0c5c45bfcc76d015a7738952ba078035ca0eda8608acecc93d4af54692",
  "data\\processed\\bhmap_bh.parquet": "8c5917c24da7e165215660d6245eb6b07988e45263f83cccbc03b73a74a4cfc6",
  "data\\processed\\lote_ctm.parquet": "58ebbd794a4f527f8018eb7aff2c1f82ff5f32f852fd3d95b89e696b18f64dcb",
  "data\\processed\\vila_favela.parquet": "f677a8b1c0996a08dacbcee253389a10d8a912b422e29d5592d19d0c047d2d91",
  "data\\processed\\declividade.parquet": "3d93aaf6c96075ea23fc96628c806f142e6320c877445f48ee766641d0f3da90",
  "data\\raw\\bhmap_context\\CADASTRO_IMOBILIARIO.csv": "563ed0fa153cfebdf43e0df48b7de2e2398ab6ee5a40fa3c9e503d59dd292b84"
 },
 "referencia": "todos os pontos de endereço do BHMap",
 "parametros": {
  "conciliacao": "limiar 90, margem 2",
  "tolerancia_lote_m": 10,
  "via_buffer_m": 25,
  "via_distancia_m": 40,
  "dispersao_ambigua_m": 100
 },
 "pares": 1062044,
 "pares_cauda":

## Conclusão

Dos registros do CNEFE, 90,0% encontram par pela chave, com mediana de 9,84 m e P90 de 35,59 m. Há 51.226 chaves exclusivas do CNEFE e 208.967 do BHMap. Mais da metade das chaves exclusivas do BHMap tem um registro do CNEFE no mesmo lote e logradouro sob outro número.

Lotes vinculados e chaves ambíguas somam metade da cauda acima de 100 m. Medir até o lote reduz a mediana à metade, e o contraste dos apartamentos cai para cerca de 1 m após estratificação por área. Exigir bairro reduz a cauda por exclusão, com perda maior nas áreas de assentamento.

As classes descrevem condições cadastrais, sem identificar causas. BHMap e IPTU compartilham a fonte municipal, e as correspondências não foram validadas independentemente.
